In [1]:
API_KEY = "nEP5mOcv27ff6yEbPzpJd2zbTyVhdEWK5boiGWHa"

In [5]:
import requests

def get_vitamin_c(food_name):

    url = "https://api.nal.usda.gov/fdc/v1/foods/search"

    params = {
        "api_key": API_KEY,
        "query": food_name,
        "pageSize": 10
    }

    headers = {
        "Accept": "application/json",
        "User-Agent": "Mozilla/5.0"
    }

    response = requests.get(url, params=params, headers=headers)

    data = response.json()

    foods = data.get("foods", [])

    if not foods:
        print("Food not found")
        return

    # Look through multiple results
    for food in foods:

        nutrients = food.get("foodNutrients", [])

        for nutrient in nutrients:

            name = nutrient.get("nutrientName", "").lower()

            if "vitamin c" in name:

                value = nutrient.get("value")
                unit = nutrient.get("unitName")

                print("Food:", food["description"])
                print("Vitamin C:", value, unit)

                return

    print("Vitamin C not found")


get_vitamin_c("Eggplant")

Food: Eggplant dip
Vitamin C: 7.1 MG


In [4]:
import requests
import json
import time
import re

# ── CONFIG ────────────────────────────────────────────────────────────────────
# API_KEY      = "YOUR_API_KEY_HERE"   # <- paste your key here
INPUT_FILE   = "foods.json"
OUTPUT_FILE  = "foods_with_usda.json"
DELAY        = 0.3   # seconds between requests
# ─────────────────────────────────────────────────────────────────────────────

SEARCH_URL = "https://api.nal.usda.gov/fdc/v1/foods/search"

HEADERS = {
    "Accept": "application/json",
    "User-Agent": "Mozilla/5.0"
}

STRIP_PATTERNS = [
    r"\braw\b", r"\bfresh\b", r"\bdried\b", r"\bcooked\b", r"\bboiled\b",
    r"\bsteamed\b", r"\bdehydrated\b", r"\buncooked\b", r"\bprepared\b",
    r"\bsprouted\b", r"\ball commercial varieties\b", r"\byear round average\b",
    r"\bincludes skin\b", r"\bwith peel\b", r"\bwith skin\b",
    r"\blow-moisture\b", r"\beuropean type\b",
]


def strip_descriptors(name):
    result = name
    for pattern in STRIP_PATTERNS:
        result = re.sub(pattern, "", result, flags=re.IGNORECASE)
    result = re.sub(r"\(.*?\)", "", result)
    result = re.sub(r",\s*,", ",", result)
    result = re.sub(r",\s*$", "", result)
    result = re.sub(r"\s{2,}", " ", result)
    return result.strip()


def query_vitc(query):
    # Returns: float if found, "" if not found, None if network error
    params = {"api_key": API_KEY, "query": query, "pageSize": 5}
    try:
        r = requests.get(SEARCH_URL, params=params, headers=HEADERS, timeout=10)
        r.raise_for_status()
        data = r.json()
    except Exception as e:
        print("  WARNING: request error for '{}': {}".format(query, e))
        return None

    for food in data.get("foods", []):
        for nutrient in food.get("foodNutrients", []):
            if "vitamin c" in nutrient.get("nutrientName", "").lower():
                value = nutrient.get("value", "")
                if value != "":
                    return value

    return ""


def get_vitamin_c(food_name):
    # 1. Full name
    result = query_vitc(food_name)
    if result:
        return result
    if result is None:
        return ""

    # 2. Strip common descriptors
    short_name = strip_descriptors(food_name)
    if short_name and short_name.lower() != food_name.lower():
        print("  retry stripped: '{}'".format(short_name))
        time.sleep(DELAY)
        result = query_vitc(short_name)
        if result:
            return result
        if result is None:
            return ""

    # 3. First word only
    first_word = short_name.split()[0] if short_name else ""
    if first_word and first_word.lower() != short_name.lower():
        print("  retry first word: '{}'".format(first_word))
        time.sleep(DELAY)
        result = query_vitc(first_word)
        if result:
            return result

    return ""


# ── Load ──────────────────────────────────────────────────────────────────────
with open(INPUT_FILE, "r", encoding="utf-8") as f:
    foods = json.load(f)

print("Loaded {} items from '{}'".format(len(foods), INPUT_FILE))

# ── Process ───────────────────────────────────────────────────────────────────
for i, item in enumerate(foods):
    if "vitC_usda" in item and item["vitC_usda"] != "":
        print("[{}/{}] skip: {}".format(i+1, len(foods), item.get("name_usda") or item.get("name", "")))
        continue

    name = item.get("name_usda") or item.get("name_en") or item.get("name", "")
    print("[{}/{}] {}".format(i+1, len(foods), name))

    vitc = get_vitamin_c(name)
    item["vitC_usda"] = vitc

    print("  -> {} mg".format(vitc) if vitc else "  -> not found")

    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        json.dump(foods, f, ensure_ascii=False, indent=2)

    time.sleep(DELAY)

# ── Done ──────────────────────────────────────────────────────────────────────
found   = sum(1 for item in foods if item.get("vitC_usda") not in ("", None))
missing = len(foods) - found
print("\nDone -> '{}'".format(OUTPUT_FILE))
print("  {} values found, {} not found".format(found, missing))


Loaded 128 items from 'foods.json'
[1/128] Garlic
  -> 80.0 mg
[2/128] Avocados
  -> 12.0 mg
[3/128] Asparagus
  -> 24.7 mg
[4/128] Eggplant
  -> not found
[5/128] Lettuce, batavia
  -> 8 mg
[6/128] Swiss chard
  -> not found
[7/128] Beets
  -> not found
[8/128] Borage
  -> not found
[9/128] Broccoli
  -> not found
[10/128] Carrots
  -> not found
[11/128] Celery
  -> 8.2 mg
[12/128] Celeriac
  -> not found
[13/128] Mushrooms, white
  -> not found
[14/128] Chicory
  -> not found
[15/128] Cabbage, Chinese (pak-choi)
  -> 45.0 mg
[16/128] Brussels sprouts
  -> 84.7 mg
[17/128] Cauliflower
  -> 21.2 mg
[18/128] Kale
  -> not found
[19/128] Cabbage, raw
  -> 36.6 mg
[20/128] Cabbage, red
  -> not found
[21/128] Cabbage, savoy
  -> 26.6 mg
[22/128] Chives
  -> not found
[23/128] Cucumber
  -> 2.5 mg
[24/128] Squash, winter
  -> not found
[25/128] Zucchini
  -> 12.6 mg
[26/128] Watercress
  -> 31.3 mg
[27/128] Endive
  -> 6.5 mg
[28/128] Spinach
  -> not found
[29/128] Beet greens
  -> not fo

In [7]:
import requests
import json

def get_food_json(food_name):
    url = "https://api.nal.usda.gov/fdc/v1/foods/search"
    params = {"api_key": API_KEY, "query": food_name, "pageSize": 3}
    headers = {"Accept": "application/json", "User-Agent": "Mozilla/5.0"}

    data = requests.get(url, params=params, headers=headers).json()
    foods = data.get("foods", [])

    if not foods:
        print("Not found")
        return

    # Print the first result in full
    print(json.dumps(foods[0], indent=2))

get_food_json("Orange")

JSONDecodeError: Expecting value: line 1 column 1 (char 0)